# Nowcasting行业景气度预测 - 研报复现

本notebook复现华泰证券金工研报《中观行业景气度：Nowcasting初探》

**数据源优先级**: efinance > akshare > baostock > tushare

**核心内容：**
1. Nowcasting模型原理与实现
2. 动态因子模型(DFM)与EM算法
3. 钢铁行业景气度指数构建
4. 方向预测准确率评估
5. 单行业择时效果回测

In [ ]:
# 导入必要的库
import sys
import os
sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath('.'))))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from source import (
    MultiSourceDataFetcher, SteelIndustryDataFetcher, DataCache, get_all_steel_indicators,
    SteelIndustrySentimentIndex, SentimentIndexComparison,
    NowcastingModel, IndicatorSelector,
    DynamicFactorModel, DFMSentimentIndex,
    IndustryTimingBacktest, GodViewBacktest, TimingComparison,
    check_stationarity, calculate_direction_accuracy,
    TARGET_INDUSTRY, START_DATE, END_DATE,
    LATENT_FACTOR_NUM, MAX_INDICATORS, OUTPUT_DIR
)

plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

print('库导入成功!')

## 1. 数据源初始化

In [ ]:
# 初始化多数据源获取器
data_fetcher = SteelIndustryDataFetcher()

print('\n数据源可用性:')
print(f'  - efinance: {data_fetcher.efinance_available}')
print(f'  - akshare: {data_fetcher.akshare_available}')
print(f'  - baostock: {data_fetcher.baostock_available}')
print(f'  - tushare: {data_fetcher.tushare_available}')

## 2. 数据获取

In [ ]:
# 获取钢铁行业数据
def generate_simulated_data():
    """生成模拟数据用于演示"""
    np.random.seed(42)
    dates = pd.date_range(start='2015-01-01', end='2023-12-31', freq='M')
    n = len(dates)
    
    data = {}
    
    latent_factor = np.cumsum(np.random.randn(n) * 0.5)
    latent_factor = (latent_factor - latent_factor.mean()) / latent_factor.std()
    
    for i in range(31):
        loading = np.random.uniform(0.3, 1.0)
        noise = np.random.randn(n) * 0.3
        indicator = loading * latent_factor + noise
        data[f'indicator_{i+1}'] = indicator
    
    return pd.DataFrame(data, index=dates)

# 尝试获取真实数据
try:
    indicator_data = data_fetcher.get_steel_indicators(START_DATE, END_DATE)
    if indicator_data is None or indicator_data.empty:
        raise ValueError('Empty data')
    print(f'使用真实数据: {indicator_data.shape}')
except Exception as e:
    indicator_data = generate_simulated_data()
    print(f'使用模拟数据进行演示 ({e})')

print(f'指标数据形状: {indicator_data.shape}')
print(f'数据时间范围: {indicator_data.index[0]} 至 {indicator_data.index[-1]}')

## 3. 指标筛选

In [ ]:
# 使用指标筛选器
selector = IndicatorSelector(
    interpretability_threshold=0.20,
    stationarity_pvalue=0.10,
    min_indicators=15
)

selected_indicators = selector.select_indicators(indicator_data)
print(f'筛选后指标数量: {len(selected_indicators.columns)}')

## 4. Nowcasting模型拟合

In [ ]:
# 构建钢铁行业景气度指数
sentiment_builder = SteelIndustrySentimentIndex(
    n_factors=LATENT_FACTOR_NUM,
    n_indicators=MAX_INDICATORS,
    interpretability_threshold=0.20,
    stationarity_pvalue=0.10
)

# 构建景气度指数
sentiment_index = sentiment_builder.build_sentiment_index(
    selected_indicators,
    use_selector=False
)

print(f'景气度指数构建完成: {len(sentiment_index)} 个数据点')

## 5. 景气度指数可视化

In [ ]:
# 绘制景气度指数
fig, ax = plt.subplots(figsize=(14, 6))

ax.plot(sentiment_index.index, sentiment_index.values, 'b-', linewidth=1.5, label='Sentiment Index')
ax.axhline(y=0, color='r', linestyle='--', alpha=0.5, label='Zero Line')

ax.set_title(f'{TARGET_INDUSTRY} 行业景气度指数 (Nowcasting)', fontsize=14)
ax.set_xlabel('日期', fontsize=12)
ax.set_ylabel('景气度指数', fontsize=12)
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'sentiment_index_plot.png'), dpi=150)
plt.show()

print(f'图表已保存至: {os.path.join(OUTPUT_DIR, "sentiment_index_plot.png")}')

## 6. 指标重要性分析

In [ ]:
# 获取指标权重
weights = sentiment_builder.get_indicator_weights()

if weights is not None and len(weights) > 0:
    top_weights = weights.nlargest(15)
    
    fig, ax = plt.subplots(figsize=(12, 6))
    top_weights.plot(kind='barh', ax=ax, color='steelblue')
    ax.set_title('Top 15 指标权重', fontsize=14)
    ax.set_xlabel('权重', fontsize=12)
    ax.set_ylabel('指标', fontsize=12)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'indicator_weights.png'), dpi=150)
    plt.show()
    
    print('\nTop 15 最重要指标:')
    print(top_weights)

## 7. 方向预测准确率评估

In [ ]:
# 生成模拟的基准序列（ROE_TTM代理）
np.random.seed(123)
n = len(sentiment_index)
benchmark = pd.Series(
    np.cumsum(np.random.randn(n) * 0.3),
    index=sentiment_index.index,
    name='benchmark'
)

# 评估方向预测准确率
accuracy_results = sentiment_builder.evaluate_direction_accuracy(benchmark)

print('方向预测准确率评估结果:')
print(f'  当前方向准确率: {accuracy_results.get("current_accuracy", 0):.2%}')
print(f'  预测方向准确率: {accuracy_results.get("predicted_accuracy", 0):.2%}')
print(f'  样本数量: {accuracy_results.get("n_samples", 0)}')

## 8. 择时回测分析

In [ ]:
# 生成模拟价格序列
np.random.seed(456)
returns = sentiment_index.pct_change().fillna(0) * 2 + np.random.randn(n) * 0.02
price_series = pd.Series(100 * (1 + returns).cumprod(), index=sentiment_index.index, name='price')

# 运行回测
backtester = IndustryTimingBacktest(initial_capital=1000000.0)
result = backtester.run_backtest(
    sentiment_index,
    price_series,
    sentiment_threshold=0.0
)

print('=' * 50)
print('回测结果汇总')
print('=' * 50)
print(f'总收益: {result.total_return:.2%}')
print(f'年化收益: {result.annual_return:.2%}')
print(f'夏普比率: {result.sharpe_ratio:.4f}')
print(f'最大回撤: {result.max_drawdown:.2%}')
print(f'胜率: {result.win_rate:.2%}')
print(f'交易次数: {result.n_trades}')

In [ ]:
# 绘制回测结果
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# 累积收益
axes[0].plot(result.cumulative_returns.index, result.cumulative_returns.values, 'b-', linewidth=1.5)
axes[0].set_title('策略累积收益', fontsize=14)
axes[0].set_xlabel('日期')
axes[0].set_ylabel('累积收益')
axes[0].grid(True, alpha=0.3)

# 超额收益
axes[1].bar(result.excess_returns.index, result.excess_returns.values, color='steelblue', alpha=0.7)
axes[1].axhline(y=0, color='r', linestyle='--', alpha=0.5)
axes[1].set_title('超额收益', fontsize=14)
axes[1].set_xlabel('日期')
axes[1].set_ylabel('超额收益')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'backtest_results.png'), dpi=150)
plt.show()

print(f'回测图表已保存至: {os.path.join(OUTPUT_DIR, "backtest_results.png")}')

## 9. Nowcasting新息分析

In [ ]:
# 获取当前和预测的景气度方向
if sentiment_builder.nowcasting_model is not None:
    nowcast_result = sentiment_builder.nowcasting_model.nowcast()
    
    print('Nowcasting结果:')
    print(f'  当前方向: {"上升" if nowcast_result.current_direction > 0 else "下降" if nowcast_result.current_direction < 0 else "持平"}')
    print(f'  预测方向: {"上升" if nowcast_result.predicted_direction > 0 else "下降" if nowcast_result.predicted_direction < 0 else "持平"}')
    print(f'  置信度: {nowcast_result.confidence:.2%}')
    print(f'  新息: {nowcast_result.new_information:.4f}')
    print(f'  预期变化: {nowcast_result.expected_change:.4f}')

## 10. 结果保存

In [ ]:
# 保存景气度指数
sentiment_path = os.path.join(OUTPUT_DIR, 'sentiment_index.csv')
sentiment_index.to_csv(sentiment_path)
print(f'景气度指数已保存至: {sentiment_path}')

# 保存回测结果
result_df = pd.DataFrame({
    'cumulative_return': result.cumulative_returns,
    'excess_return': result.excess_returns
})
result_df.to_csv(os.path.join(OUTPUT_DIR, 'backtest_results.csv'))
print(f'回测结果已保存至: {os.path.join(OUTPUT_DIR, "backtest_results.csv")}')

## 总结

本notebook完成了以下内容：

1. **多数据源集成**: 集成efinance、akshare、baostock、tushare四大数据源
2. **数据获取**: 优先使用真实数据，失败时自动使用模拟数据
3. **指标筛选**: 基于平稳性和解释度筛选代理指标
4. **Nowcasting模型**: 实现动态因子模型，使用EM算法估计隐含因子
5. **景气度指数构建**: 合成行业景气度指数
6. **回测分析**: 评估基于景气度指数的择时效果

**注意**：如需完整复现研报结果，请确保数据源可用并获取真实的钢铁行业数据。当前使用模拟数据进行演示验证。